# Auditoria BCBI × SAGI (Relatório Entrada/Saída)

Compara os KPIs do **BCBI** com as somas dos relatórios **sem agrupamento** do SAGI:
`relatorio entrada.csv` e `relatorio saida.csv` em `02-Referencias/`.

**Divisão:** Seletiva · **Período:** mês corrente (automático)

1. Filtre o BCBI (Seletiva + mês corrente).
2. Preencha o dicionário `bcbi` com os valores do dashboard.
3. Execute todas as células (`Run All`).

Ajuste `TOLERANCIA` na primeira célula de código se precisar.

In [13]:
# Imports e configuração
import os
import pandas as pd
from datetime import datetime
from IPython.display import display, HTML

_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, '02-Referencias')):
    WORKSPACE = _cwd
elif os.path.isdir(os.path.join(_cwd, '..', '02-Referencias')):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, '..'))
elif os.path.isdir(os.path.join(_cwd, '..', '..', '02-Referencias')):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, '..', '..'))
else:
    raise FileNotFoundError(f'Pasta 02-Referencias não encontrada a partir de: {_cwd}')

REFS = os.path.join(WORKSPACE, '02-Referencias')
PATH_ENTRADA_REL = os.path.join(REFS, 'relatorio entrada.csv')
PATH_SAIDA_REL = os.path.join(REFS, 'relatorio saida.csv')

hoje = datetime.now()
MES = f'{hoje.year}-{hoje.month:02d}'
MES_LABEL = hoje.strftime('%m/%Y')

TOLERANCIA = 1_000.00

print(f'Período de análise : {MES_LABEL}  ({MES})')
print(f'Tolerância         : R$ {TOLERANCIA:,.2f}')
print(f'REL ENTRADA SAGI   : {PATH_ENTRADA_REL}')
print(f'REL SAÍDA SAGI     : {PATH_SAIDA_REL}')

Período de análise : 05/2026  (2026-05)
Tolerância         : R$ 1,000.00
REL ENTRADA SAGI   : c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\relatorio entrada.csv
REL SAÍDA SAGI     : c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\relatorio saida.csv


In [14]:
# Preencha com os valores exibidos no BCBI (Seletiva, mês corrente)

bcbi = {
    'receitas_totais_rs':    252.45,
    'vendas_sucata_rs':      0.00,
    'vendas_sucata_kg':      0,
    'compras_financeiro_rs': 3_886.90,
    'compras_sucata_rs':     3_886.90,
    'compras_sucata_kg':     5_130,
}

print(f"BCBI — {len(bcbi)} métricas · {hoje.strftime('%d/%m/%Y')}")

BCBI — 6 métricas · 04/05/2026


In [15]:
def _parse_relatorio_sagi_sem_agrupamento(path_csv, tipo):
    """Extrai linhas de detalhe do relatório textual exportado pelo SAGI."""
    rows = []
    with open(path_csv, 'r', encoding='latin-1', errors='ignore') as f:
        for raw in f:
            line = raw.rstrip('\n')
            if not line or ';' not in line:
                continue
            parts = [p.strip() for p in line.split(';')]
            if not parts or not parts[0].isdigit():
                continue
            vals = [p for p in parts if p]
            if len(vals) < 10:
                continue
            idx_data = next(
                (i for i, v in enumerate(vals) if len(v) == 10 and v[2] == '/' and v[5] == '/'),
                None,
            )
            if idx_data is None or idx_data + 8 >= len(vals):
                continue
            rows.append({
                'tipo': tipo,
                'boleto': vals[0],
                'data': vals[idx_data],
                'pessoa': vals[idx_data + 1],
                'quantidade_kg': vals[idx_data + 2],
                'valor_total': vals[idx_data + 4],
                'produto': vals[idx_data + 7],
            })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out['data'] = pd.to_datetime(out['data'], format='%d/%m/%Y', errors='coerce')
    for col in ('quantidade_kg', 'valor_total'):
        out[col] = (
            out[col].astype(str)
            .str.replace(r'[^0-9,.-]', '', regex=True)
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['filial'] = 'Consolidado SAGI'
    return out


df_entrada_rel = _parse_relatorio_sagi_sem_agrupamento(PATH_ENTRADA_REL, 'ENTRADA')
df_entrada_rel = df_entrada_rel[df_entrada_rel['data'].dt.strftime('%Y-%m').eq(MES)].copy()
_prod_ent = df_entrada_rel['produto'].astype(str).str.upper()
df_entrada_rel_sucata = df_entrada_rel[_prod_ent.str.contains('SUCATA', na=False)].copy()

df_saida_rel = _parse_relatorio_sagi_sem_agrupamento(PATH_SAIDA_REL, 'SAIDA')
df_saida_rel = df_saida_rel[df_saida_rel['data'].dt.strftime('%Y-%m').eq(MES)].copy()
_prod = df_saida_rel['produto'].astype(str).str.upper()
_mask_vendas_suc = _prod.str.contains('FERRO', na=False) | _prod.str.contains('PROCESS', na=False)
df_saida_rel_sucata = df_saida_rel[_mask_vendas_suc].copy()

odbc_receitas_totais = df_saida_rel['valor_total'].sum()
odbc_vendas_sucata_rs = df_saida_rel_sucata['valor_total'].sum()
odbc_vendas_sucata_kg = df_saida_rel_sucata['quantidade_kg'].sum()
odbc_compras_fin_rs = df_entrada_rel['valor_total'].sum()
odbc_compras_sucata_rs = df_entrada_rel_sucata['valor_total'].sum()
odbc_compras_sucata_kg = df_entrada_rel_sucata['quantidade_kg'].sum()

print(
    f"SAGI Relatórios — {MES_LABEL} · "
    f"entrada {len(df_entrada_rel)} · saída {len(df_saida_rel)} · "
    f"compra sucata {len(df_entrada_rel_sucata)} · venda sucata {len(df_saida_rel_sucata)}"
)

SAGI Relatórios — 05/2026 · entrada 7 · saída 1 · compra sucata 7 · venda sucata 0


In [16]:
_NA = float('nan')


def _fmtv(v, kg=False):
    if pd.isna(v):
        return '—'
    return f'{v:,.0f} kg' if kg else f'R$ {v:,.2f}'


def _fmtd(v):
    if pd.isna(v):
        return '—'
    return f'{v:+,.2f}'


def _fmtp(diff, base):
    if pd.isna(diff) or pd.isna(base) or base == 0:
        return '—'
    return f'{diff / base * 100:+.2f}%'


def _severity(v):
    if pd.isna(v):
        return 'none'
    if abs(v) <= TOLERANCIA:
        return 'ok'
    if abs(v) <= 10_000:
        return 'warn'
    return 'error'


def _severity_pct(diff, base, pct_ok=0.01):
    if pd.isna(diff):
        return 'none'
    if not (pd.isna(base) or base == 0) and abs(diff / base) < pct_ok:
        return 'ok'
    return _severity(diff)


_ICON = {'error': '❌', 'warn': '⚠️', 'ok': '✅', 'none': '∅'}


def _cor_delta(v):
    if pd.isna(v):
        return 'background-color:#e9ecef;color:#6c757d'
    if abs(v) <= TOLERANCIA:
        return 'background-color:#198754;color:#ffffff;font-weight:bold'
    if abs(v) <= 10_000:
        return 'background-color:#fd7e14;color:#ffffff;font-weight:bold'
    return 'background-color:#dc3545;color:#ffffff;font-weight:bold'


def _cor_pct(v):
    if v == '—':
        return 'background-color:#e9ecef;color:#6c757d'
    try:
        n = abs(float(v.replace('%', '').replace('+', '').replace('-', '')))
        if n <= 0.5:
            return 'color:#198754;font-weight:bold'
        if n <= 5.0:
            return 'color:#fd7e14;font-weight:bold'
        return 'color:#dc3545;font-weight:bold'
    except ValueError:
        return ''


def _cor_empty(v):
    if v == '—':
        return 'background-color:#e9ecef;color:#6c757d'
    return ''


linhas = [
    ('Receitas Totais (R$)', bcbi['receitas_totais_rs'], odbc_receitas_totais, False),
    ('Vendas Sucata (R$)', bcbi['vendas_sucata_rs'], odbc_vendas_sucata_rs, False),
    ('Vendas Sucata (KG)', bcbi['vendas_sucata_kg'], odbc_vendas_sucata_kg, True),
    ('Compras Financeiro (R$)', bcbi['compras_financeiro_rs'], odbc_compras_fin_rs, False),
    ('Compras Sucata (R$)', bcbi['compras_sucata_rs'], odbc_compras_sucata_rs, False),
    ('Compras Sucata (KG)', bcbi['compras_sucata_kg'], odbc_compras_sucata_kg, True),
]

rows = []
for label, b, o, kg in linhas:
    d = b - o if not (pd.isna(b) or pd.isna(o)) else _NA
    sev = _severity_pct(d, o, pct_ok=0.01)
    rows.append({
        'Status': _ICON[sev],
        'Métrica': label,
        'BCBI': _fmtv(b, kg),
        'SAGI (Rel. Entrada/Saída)': _fmtv(o, kg),
        'Δ BCBI-Relatório': d,
        '% Relatório': _fmtp(d, o),
    })

cmp = pd.DataFrame(rows)

_TH = (
    'background-color:#1f3864;color:#ffffff;font-weight:600;'
    'padding:9px 13px;text-align:center;white-space:nowrap;font-size:13px'
)
_TD = 'padding:7px 12px;border:1px solid #dee2e6;font-size:13px;text-align:right'
_TDL = 'padding:7px 12px;border:1px solid #dee2e6;font-size:13px;text-align:left;font-weight:500'
_TDC = 'padding:7px 8px;border:1px solid #dee2e6;font-size:15px;text-align:center'

styled = (
    cmp.style.format({'Δ BCBI-Relatório': _fmtd}, na_rep='—').set_table_styles(
        [
            {'selector': 'table', 'props': 'border-collapse:collapse;width:100%;font-family:system-ui,-apple-system,sans-serif'},
            {'selector': 'th', 'props': _TH},
            {'selector': 'td', 'props': _TD},
            {'selector': 'td:nth-child(1)', 'props': _TDC},
            {'selector': 'td:nth-child(2)', 'props': _TDL},
            {'selector': 'td:nth-child(3)', 'props': _TD + ';font-weight:600'},
            {'selector': 'tr:nth-child(even)', 'props': 'background-color:#f8f9fa'},
            {'selector': 'tr:hover td', 'props': 'filter:brightness(0.94)'},
        ]
    )
)

try:
    styled = styled.map(_cor_delta, subset=['Δ BCBI-Relatório'])
    styled = styled.map(_cor_pct, subset=['% Relatório'])
    styled = styled.map(_cor_empty, subset=['SAGI (Rel. Entrada/Saída)'])
except AttributeError:
    styled = styled.applymap(_cor_delta, subset=['Δ BCBI-Relatório'])
    styled = styled.applymap(_cor_pct, subset=['% Relatório'])
    styled = styled.applymap(_cor_empty, subset=['SAGI (Rel. Entrada/Saída)'])

_COLS_TEXTO = ['Métrica', 'BCBI', 'SAGI (Rel. Entrada/Saída)']


def _cor_texto_fundo_claro(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for idx in df.index:
        if idx % 2 == 1:
            for col in _COLS_TEXTO:
                styles.at[idx, col] = 'color:#181818'
    return styles


styled = styled.apply(_cor_texto_fundo_claro, axis=None)
display(styled)

n_err = cmp['Status'].eq(_ICON['error']).sum()
n_warn = cmp['Status'].eq(_ICON['warn']).sum()
n_ok = cmp['Status'].eq(_ICON['ok']).sum()
n_none = cmp['Status'].eq(_ICON['none']).sum()

summary = (
    f'<span style="color:#198754;font-weight:bold">✅ {n_ok} OK</span>&nbsp;&nbsp;|&nbsp;&nbsp;'
    f'<span style="color:#fd7e14;font-weight:bold">⚠️ {n_warn} Atenção</span>&nbsp;&nbsp;|&nbsp;&nbsp;'
    f'<span style="color:#dc3545;font-weight:bold">❌ {n_err} Divergência</span>&nbsp;&nbsp;|&nbsp;&nbsp;'
    f'<span style="color:#6c757d">∅ {n_none} Sem fonte</span>'
)

div_bo = cmp[cmp['Status'].isin([_ICON['warn'], _ICON['error']])]
total_div = len(div_bo)
now_str = hoje.strftime('%d/%m/%Y %H:%M')

if total_div == 0:
    box = (
        'background:linear-gradient(120deg,#d4edda,#c3e6cb);color:#155724;'
        'padding:16px 20px;border-radius:10px;border-left:5px solid #198754;'
        'font-size:1.05em;font-weight:bold;margin-top:14px'
    )
    display(
        HTML(
            f'<div style="{box}">'
            f'✅  AUDITORIA OK — Todos os valores batem (tolerância R$ {TOLERANCIA:,.0f})'
            f'&nbsp;&nbsp;|&nbsp;&nbsp;{now_str}'
            f'<div style="font-weight:normal;font-size:0.88em;margin-top:6px">{summary}</div>'
            f'</div>'
        )
    )
else:
    box = (
        'background:linear-gradient(120deg,#f8d7da,#f5c6cb);color:#721c24;'
        'padding:16px 20px;border-radius:10px;border-left:5px solid #dc3545;'
        'font-size:1.05em;font-weight:bold;margin-top:14px'
    )
    items_html = ''
    for _, row in div_bo.iterrows():
        items_html += (
            f'<li><b>{row["Métrica"]}</b> (BCBI vs Relatório SAGI): '
            f'Δ = {_fmtd(row["Δ BCBI-Relatório"])} &nbsp; {row["% Relatório"]}</li>'
        )
    display(
        HTML(
            f'<div style="{box}">'
            f'⚠️  DIVERGÊNCIAS ENCONTRADAS: {total_div} métrica(s)'
            f'&nbsp;&nbsp;|&nbsp;&nbsp;{now_str}'
            f'<ul style="font-weight:normal;margin-top:8px;margin-bottom:8px">{items_html}</ul>'
            f'<div style="font-weight:normal;font-size:0.88em;border-top:1px solid #f5c6cb;padding-top:6px">{summary}</div>'
            f'</div>'
        )
    )

,Status,Métrica,BCBI,SAGI (Rel. Entrada/Saída),Δ BCBI-Relatório,% Relatório
0,✅,Receitas Totais (R$),R$ 252.45,R$ 252.45,+0.00,+0.00%
1,✅,Vendas Sucata (R$),R$ 0.00,R$ 0.00,+0.00,—
2,✅,Vendas Sucata (KG),0 kg,0 kg,+0.00,—
3,✅,Compras Financeiro (R$),"R$ 3,886.90","R$ 3,886.90",+0.00,+0.00%
4,✅,Compras Sucata (R$),"R$ 3,886.90","R$ 3,886.90",+0.00,+0.00%
5,✅,Compras Sucata (KG),"5,130 kg","5,130 kg",+0.00,+0.00%
